# Week 21, Day 4: Systems of linear equations and the matrix exponential

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w021/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Three ways to the same answer

For the 3 × 3 matrix A, write `expm_series(A, t)` (sum terms until they are below 1e-15) and `expm_eig(A, t)` (V exp(Λt) V⁻¹, using np.linalg.eig and taking the real part). Compare both, and compare e^{At}x0 with an RK4 simulation `x_sim` at t = 1.5 using 1,500 steps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
A = np.array([[-1.0, 2.0, 0.0], [-2.0, -1.0, 0.5], [0.0, 0.3, -0.5]])
x0 = np.array([1.0, 0.0, -1.0])

def expm_series(A, t):
    term = np.eye(len(A)); total = term.copy(); k = 1
    while np.abs(term).max() > 1e-15:
        term = ...                                 # next term (At)^k / k!
        total = total + term; k += 1
    return total

def expm_eig(A, t):
    lam, V = np.linalg.eig(A)
    return ...

x, dt = x0.copy(), 1.5 / 1500
f = lambda x: A @ x
for _ in range(1500):
    k1 = f(x); k2 = f(x + dt / 2 * k1); k3 = f(x + dt / 2 * k2); k4 = f(x + dt * k3)
    x = x + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
x_sim = x
print("eigenvalues:", np.linalg.eigvals(A).round(3)); print("e^{At} x0:", (expm_eig(A, 1.5) @ x0).round(6), " simulated:", x_sim.round(6))

In [ ]:
import numpy as np
def test_eig():
    "expm_eig agrees with the power series to 1e-9"
    assert np.max(np.abs(expm_eig(A, 1.5) - expm_series(A, 1.5))) < 1e-9

def test_sim():
    "e^{At} x0 agrees with an RK4 simulation to 1e-6"
    assert np.max(np.abs(expm_eig(A, 1.5) @ x0 - x_sim)) < 1e-6

test_eig()
test_sim()
print("All checks passed")

### Exercise: Two time constants from one recording

For a soma (τ = 10 ms) and dendrite (τ = 30 ms) coupled with strength g = 0.5, build the matrix A2, find the fast and slow time constants from its eigenvalues, then compute the soma voltage after a pulse with the matrix exponential (0 to 150 ms) and fit the slope of log V_soma over 60 to 150 ms to get `tau_late`.

In [ ]:
tS, tD, g = 10.0, 30.0, 0.5
A2 = np.array([[-(1 + g) / tS, g / tS],
               [g / tD, -(1 + g) / tD]])
lam2 = np.sort(np.linalg.eigvals(A2).real)
tau_fast, tau_slow = ..., ...
ts = np.arange(0, 150, 0.5)
V_soma = np.array([(expm_eig(A2, t) @ np.array([1.0, 0.0]))[0] for t in ts])
late = ts >= 60
tau_late = ...
print(f"fast mode {tau_fast:.2f} ms, slow mode {tau_slow:.2f} ms;  late slope gives {tau_late:.2f} ms")
plt.semilogy(ts, V_soma); plt.xlabel("ms"); plt.ylabel("soma voltage (log)"); plt.show()

In [ ]:
import numpy as np
def test_modes():
    "tau_fast and tau_slow are the two time constants from the eigenvalues of A2"
    lam = np.sort(np.linalg.eigvals(A2).real)
    assert abs(tau_fast + 1 / lam[0]) < 1e-9 and abs(tau_slow + 1 / lam[1]) < 1e-9

def test_peel():
    "the late slope of log V_soma recovers the slow time constant to within 3%"
    assert abs(tau_late - tau_slow) / tau_slow < 0.03

test_modes()
test_peel()
print("All checks passed")

### Exercise: The circular law and stability

For N = 200 and g = 0.8 and 1.2, draw W with entries of standard deviation g/√N, store its eigenvalues in `eigs[g]`, and simulate ẋ = −x + Wx from a random start for 40 time units with Euler steps of 0.05, recording the norm of x at each step in `norms[g]`.

In [ ]:
rng = np.random.default_rng(2006)
N = 200
eigs, norms = {}, {}
for g in (0.8, 1.2):
    W = rng.normal(0, g / np.sqrt(N), (N, N))
    eigs[g] = np.linalg.eigvals(W)
    x = rng.normal(size=N); nr = [np.linalg.norm(x)]
    for _ in range(800):
        x = ...
        nr.append(np.linalg.norm(x))
    norms[g] = np.array(nr)
print({g: round(float(np.abs(e).max()), 3) for g, e in eigs.items()}, " largest eigenvalue modulus")
print({g: f"{n[-1] / n[0]:.3g}" for g, n in norms.items()}, " final / initial norm")
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for g in eigs: ax[0].plot(eigs[g].real, eigs[g].imag, ".", ms=3, label=f"g = {g}")
ax[0].axvline(1, c="k", ls="--"); ax[0].set_aspect("equal"); ax[0].legend()
for g in norms: ax[1].semilogy(norms[g], label=f"g = {g}")
ax[1].legend(); plt.show()

In [ ]:
import numpy as np
def test_disc():
    "the largest eigenvalue modulus is within 10% of g"
    assert abs(np.abs(eigs[0.8]).max() - 0.8) < 0.08 and abs(np.abs(eigs[1.2]).max() - 1.2) < 0.12

def test_dynamics():
    "activity decays for g = 0.8 and grows for g = 1.2"
    assert norms[0.8][-1] < 0.01 * norms[0.8][0] and norms[1.2][-1] > 10 * norms[1.2][0]

test_disc()
test_dynamics()
print("All checks passed")